# Building a Self-Improving Morning Brief

**Workshop Goal**: Build a morning brief generator that learns and improves over time

**Duration**: 60 minutes

**What You'll Build**:
- Morning brief generator for GitHub repos you follow
- Agent that remembers your preferences
- System that learns from your feedback
- Self-deploying automation

---

## Before You Start

**In a terminal, start the vLLM server:**

```bash
VLLM_USE_TRITON_FLASH_ATTN=0 vllm serve cerebras/MiniMax-M2.5-REAP-139B-A10B \
    --served-model-name MiniMax-M2.5 \
    --api-key abc-123 \
    --port 8000 \
    --enable-auto-tool-choice \
    --tool-call-parser minimax_m2 \
    --trust-remote-code \
    --reasoning-parser minimax_m2_append_think \
    --max-model-len 160000 \
    --gpu-memory-utilization 0.99
```

This will take 2-3 minutes to load the 139B parameter model into GPU memory.

---

## The Journey

1. **Intro to Mini-Agent** (10 min)
2. **Build Morning Brief** (20 min)
3. **Add Self-Improvement** (25 min)
4. **The Challenge** (5 min)

In [ ]:
# Environment configuration
import os

BASE_URL = "http://localhost:8000/v1"
API_KEY = "abc-123"
MODEL = "MiniMax-M2.5"

os.environ["BASE_URL"] = BASE_URL
os.environ["OPENAI_API_KEY"] = API_KEY

print(f"Configuration:")
print(f"  Endpoint: {BASE_URL}")
print(f"  Model: {MODEL}")
print(f"\nReady to connect!")

In [ ]:
# Verify vLLM server is running
!curl http://localhost:8000/v1/models -H "Authorization: Bearer $OPENAI_API_KEY"

In [ ]:
# Clone Mini-Agent repository and install
import os
import sys

# Clone the repo if not already cloned
if not os.path.exists("Mini-Agent"):
    !git clone https://github.com/MiniMax-AI/Mini-Agent.git
    print("✓ Repository cloned!")
else:
    print("✓ Repository already exists")

# Install Mini-Agent
!cd Mini-Agent && pip install -q -e .

# Add to Python path
mini_agent_path = os.path.abspath("Mini-Agent")
if mini_agent_path not in sys.path:
    sys.path.insert(0, mini_agent_path)

print("✓ Mini-Agent installed!")
print(f"✓ Path added: {mini_agent_path}")
print()
print("Ready to start the workshop!")

---

# Part 1: Introduction to Mini-Agent

## What is Mini-Agent?

Mini-Agent is an AI agent framework that:
- Runs with **any OpenAI-compatible API** (local or cloud)
- Works with **any model** (open or commercial)
- Has powerful **tools** (file operations, bash, memory)
- Enables **autonomous learning** through Session Notes

## Today's Workshop Setup

**We're using:**
```
Model: MiniMax M2.5 (139B parameters)
Deployment: vLLM on AMD MI300X GPU
Endpoint: http://localhost:8000/v1
Cost: $0 (running locally)
```

**Why this setup?**
- No API costs after deployment
- Full data privacy
- High performance (MI300X with 192GB HBM)

**But you can use:**
- OpenAI API
- Anthropic API
- Any vLLM deployment
- Any OpenAI-compatible endpoint

Let's test the connection!

In [ ]:
import sys
from pathlib import Path

# Add Mini-Agent to path
from mini_agent import Agent, LLMClient, Message
from mini_agent.schema import LLMProvider

# Configuration for our workshop endpoint
llm_client = LLMClient(
    api_key="abc-123",
    provider=LLMProvider.OPENAI,
    api_base="http://localhost:8000/v1",
    model="MiniMax-M2.5"
)

# Test connection
print("Testing connection...")
test_messages = [Message(role="user", content="Respond with 'Connection successful!' if you can read this.")]

try:
    response = await llm_client.generate(test_messages)
    print(f"✓ {response.content[:100]}")
    print("\nReady to build!")
except Exception as e:
    print(f"✗ Connection failed: {e}")

## Mini-Agent's Key Features

**1. File Operations** - Read, write, edit files

**2. Bash Commands** - Execute shell commands

**3. Session Notes** - Persistent memory across sessions

We'll use all three to build our morning brief generator!

---

# Part 2: Building the Morning Brief

## The Problem

As an AI/ML developer, you follow tons of GitHub repos:
- vLLM (inference optimization)
- PyTorch (ML framework)
- Mini-Agent (this project!)
- Transformers (model library)
- And many more...

Too many to check daily!

## The Solution

Build an AI agent that:
1. Checks repos for updates
2. Filters based on your interests
3. Generates a personalized brief
4. Runs automatically

Let's build it together!

## Step 1: Setup Tools and Workspace

In [ ]:
from mini_agent.tools import BashTool, ReadTool, WriteTool

# Create workspace
workspace = Path("./workspace")
workspace.mkdir(exist_ok=True)

# Basic tools for our morning brief
tools = [
    BashTool(workspace_dir=str(workspace)),
    ReadTool(),
    WriteTool()
]

print("Workspace ready")
print(f"Location: {workspace.absolute()}")
print("\nTools: BashTool, ReadTool, WriteTool")

## Step 2: Create the Brief Generator Agent

In [ ]:
# System prompt for the basic agent
system_prompt = """You are a helpful AI assistant that generates morning briefs for developers.

Your job is to:
1. Check GitHub repositories for recent activity
2. Analyze commits and changes
3. Filter based on user preferences
4. Generate clear, actionable summaries

Use the available tools to accomplish your tasks."""

# Create agent with our tools
brief_agent = Agent(
    llm_client=llm_client,
    system_prompt=system_prompt,
    tools=tools,
    max_steps=50,
    workspace_dir=str(workspace)
)

print("Morning Brief Agent created!")
print("The agent can use git, analyze commits, and generate briefs.")

## Step 3: Generate Our First Brief!

Let's ask the agent to create a morning brief for the **vLLM** repository.

In [8]:
print("📰 Generating morning brief for vLLM repository...")
print("=" * 60)
print()

task = """
Create a morning brief about recent changes in the vLLM GitHub repository.

Repository: https://github.com/vllm-project/vllm

My interests:
- Performance improvements and optimizations
- GPU-related updates
- New features

What to skip:
- Documentation-only changes
- Minor typo fixes

Format:
- Show last 5 relevant commits
- Brief summary of each
- Why it matters

Save the brief to 'morning_brief.md'
"""

brief_agent.add_user_message(task)
result = await brief_agent.run()

📰 Generating morning brief for vLLM repository...

📝 Log file: /home/mahdi/.mini-agent/log/agent_run_20260223_121326.log

╭──────────────────────────────────────────────────────────╮
│ 💭 Step 1/50                                             │
╰──────────────────────────────────────────────────────────╯

🤖 Assistant:
<think>The user wants me to create a morning brief about recent changes in a GitHub repository. Let me first understand what I need to do:

1. Check the GitHub repository for recent activity
2. Analyze commits and changes
3. Filter based on preferences (performance, GPU, new features)
4. Skip documentation-only changes and minor typo fixes
5. Save to 'morning_brief.md'

I need to use git commands to fetch information about the repository. Let me start by checking if this is a git repository and getting the recent commits.
</think>

I'll create a morning brief for you about the vLLM repository. Let me first check out the repository and analyze recent commits.


🔧 Tool Call: 

## Check the Result

In [ ]:
brief_file = workspace / "morning_brief.md"

if brief_file.exists():
    print("Morning brief generated!")
    print("=" * 60)
    print(brief_file.read_text())
    print("=" * 60)
    print("\nSuccess! We built a working morning brief generator!")
else:
    print("Brief not created - check the agent output above")

---

# Part 3: Adding Self-Improvement

## The Challenge

Our brief generator works! But...

**What if tomorrow you want:**
- To check the same repos without re-explaining?
- To add new preferences?
- The agent to remember what you care about?

**Right now**: You'd have to repeat all your preferences every time.

## The Solution: Session Notes

Session Notes gives agents persistent memory:
- **record_note(category, content)** - Save information
- **recall_notes(category)** - Retrieve information
- **Categories** - Organize knowledge
- **Agent decides** - What to remember and when

Let's add this capability!

## Step 1: Add Session Notes Tools

In [ ]:
from mini_agent.tools import SessionNoteTool, RecallNoteTool
import json

# Memory file for the agent
memory_file = workspace / ".memory.json"

# Enhanced tools with memory
tools_with_memory = [
    SessionNoteTool(memory_file=str(memory_file)),
    RecallNoteTool(memory_file=str(memory_file)),
    BashTool(workspace_dir=str(workspace)),
    ReadTool(),
    WriteTool()
]

print("Session Notes enabled!")
print("\nThe agent can now:")
print("  • Remember your repos")
print("  • Remember your preferences")
print("  • Learn from feedback")
print("  • Improve over time")

## Step 2: Create Self-Improving Agent

We'll add a **system prompt** that teaches the agent to use memory.

In [ ]:
system_prompt = """You are a personalized morning brief generator for AI/ML developers.

MEMORY MANAGEMENT:

1. At the START of every conversation:
   - Use recall_notes to check for saved information
   - Look for: repos to track, user preferences, past feedback
   - If you find saved info, use it automatically!

2. When user shares preferences or repos:
   - Use record_note to save for future sessions
   - Be smart about categorization:
     • 'repos' - GitHub repositories to track
     • 'preferences' - What to show/skip, interests
     • 'feedback' - User improvements and requests
     • 'format' - How user likes briefs formatted

3. Build knowledge over time:
   - Each session should ADD to what you know
   - Apply ALL past preferences + new feedback
   - Get better with each interaction

Your goal: Generate briefs that improve through learning!
"""

# Create the self-improving agent
learning_agent = Agent(
    llm_client=llm_client,
    system_prompt=system_prompt,
    tools=tools_with_memory,
    max_steps=50,
    workspace_dir=str(workspace)
)

print("Self-Improving Agent created!")
print("This agent will remember repos, preferences, and learn from feedback.")

## Step 3: Session 1 - Agent Learns Your Preferences

In [ ]:
# Clear memory for demo
if memory_file.exists():
    memory_file.unlink()

print("SESSION 1: Teaching the Agent")
print("=" * 60)
print()
print("Watch for:")
print("   1. Agent recalls notes (finds nothing - first time)")
print("   2. Agent generates the brief")
print("   3. Agent RECORDS your preferences")
print()

task = """
Create a morning brief for the vLLM repository.

My preferences:
- I care about performance improvements and GPU optimization
- Skip documentation changes and minor fixes
- Focus on features that impact inference speed

Save to 'brief_v1.md'
"""

learning_agent.add_user_message(task)
result = await learning_agent.run()

## Check What the Agent Remembered

In [ ]:
if memory_file.exists():
    print("What the Agent Remembered:")
    print("=" * 60)
    
    memory = json.loads(memory_file.read_text())
    
    for i, note in enumerate(memory, 1):
        print(f"\n{i}. [{note.get('category', 'uncategorized')}]")
        print(f"   {note['content']}")
    
    print("\n" + "=" * 60)
    print("\nThe agent AUTONOMOUSLY decided to save this!")
    print("We didn't explicitly say 'remember this'")
else:
    print("No memory file created")

## Step 4: Session 2 - Agent Recalls and Applies Feedback

Now let's create a **brand new agent** (simulating a new day) and give it feedback.

**Watch**: The agent will recall all preferences and apply new feedback!

In [ ]:
# Brand new agent - simulates closing terminal and coming back tomorrow
learning_agent_day2 = Agent(
    llm_client=llm_client,
    system_prompt=system_prompt,
    tools=tools_with_memory,
    max_steps=50,
    workspace_dir=str(workspace)
)

print("SESSION 2: New Day, New Agent")
print("=" * 60)
print()
print("This is a BRAND NEW agent instance (zero conversation history)")
print()
print("Watch for:")
print("   1. Agent recalls notes from yesterday")
print("   2. Agent uses OLD preferences automatically")
print("   3. Agent applies NEW feedback on top")
print("   4. Agent records the new feedback")
print()
print("This is the self-improvement in action!")
print()

task2 = """
Generate my morning brief.

New feedback:
- Add commit hashes for each change
- Show the author name
- Prioritize breaking changes

Save to 'brief_v2.md'
"""

learning_agent_day2.add_user_message(task2)
result2 = await learning_agent_day2.run()

## See the Cumulative Learning

In [ ]:
print("What Just Happened?")
print("=" * 60)
print()
print("The agent:")
print("  ✓ Remembered vLLM repo")
print("  ✓ Remembered preferences")
print("  ✓ Applied NEW feedback")
print("  ✓ Recorded the new feedback")
print()
print("Updated Memory:")
print("=" * 60)

memory = json.loads(memory_file.read_text())
for i, note in enumerate(memory, 1):
    print(f"\n{i}. [{note.get('category', 'uncategorized')}]")
    print(f"   {note['content']}")

print("\n" + "=" * 60)
print("\nKnowledge is CUMULATIVE!")
print("Each session BUILDS on the last.")

## Compare the Briefs

In [ ]:
print("Evolution of Your Brief:")
print("=" * 60)

v1 = workspace / "brief_v1.md"
v2 = workspace / "brief_v2.md"

if v1.exists():
    print("\nVERSION 1 (Session 1):")
    print("-" * 60)
    print(v1.read_text()[:500] + "...\n")

if v2.exists():
    print("\nVERSION 2 (Session 2 - Improved):")
    print("-" * 60)
    print(v2.read_text()[:500] + "...\n")

print("=" * 60)
print()
print("V2 includes everything from V1 PLUS new feedback.")
print("The agent is learning!")

## How Session Notes Works

Let's look under the hood:

In [ ]:
print("Session Notes Architecture")
print("=" * 60)
print()
print("THREE COMPONENTS:")
print()
print("1. TOOLS")
print("   • record_note(category, content) - Saves to JSON")
print("   • recall_notes(category) - Retrieves from JSON")
print()
print("2. SYSTEM PROMPT")
print("   • Instructs agent WHEN to record")
print("   • Instructs agent WHEN to recall")
print("   • Suggests category organization")
print()
print("3. AGENT AUTONOMY")
print("   • Agent decides WHAT to remember")
print("   • Agent chooses categories")
print()
print("=" * 60)
print()
print(f"Storage: Simple JSON file at {memory_file}")
print("You can edit it, version control it, or extend to a database.")

---

# Part 4: The Challenge

**Time:** 10 minutes
**Winners:** Top 5-10

---

## Your Mission

**Can you get your agent to run automatically at a set time every morning... just by asking it?**

That's it. Figure out the rest.

---

**To win, show us:**
1. Your prompt
2. Proof it's scheduled

---

**Go!**

In [ ]:
# YOUR CHALLENGE WORKSPACE
# Make the agent run automatically at a set time every morning

# Create an agent for this challenge
challenge_agent = Agent(
    llm_client=llm_client,
    system_prompt=system_prompt,
    tools=tools_with_memory,  # Agent has BashTool, ReadTool, WriteTool, Session Notes
    max_steps=50,
    workspace_dir=str(workspace)
)

print("Challenge Agent Ready!")
print("\nThe agent has access to:")
print("  • BashTool - Can run any shell command")
print("  • WriteTool - Can create files")
print("  • Session Notes - Knows your preferences")
print()
print("Your mission: Prompt it to deploy itself at a set time.")
print()

# YOUR PROMPT HERE:
deployment_prompt = """
[WRITE YOUR PROMPT - How do you ask the agent to run automatically every morning at 8am?]
"""

# When ready, run these lines:
challenge_agent.add_user_message(deployment_prompt)
result = await challenge_agent.run()

# Then show the presenter - they'll verify with:
# !crontab -l

## Optional: Build Your Own Custom Brief

If you finish the challenge early or want to experiment more,
customize everything below for your own use case!

In [ ]:
# Your memory file
my_memory_file = workspace / ".my_memory.json"

# Clear for fresh start
if my_memory_file.exists():
    my_memory_file.unlink()

# Your tools
my_tools = [
    SessionNoteTool(memory_file=str(my_memory_file)),
    RecallNoteTool(memory_file=str(my_memory_file)),
    BashTool(workspace_dir=str(workspace)),
    ReadTool(),
    WriteTool()
]

# Your agent
my_agent = Agent(
    llm_client=llm_client,
    system_prompt=system_prompt,
    tools=my_tools,
    max_steps=50,
    workspace_dir=str(workspace)
)

print("YOUR CHALLENGE WORKSPACE")
print("=" * 60)
print()
print("Customize your brief:")
print("  • Which repos?")
print("  • What topics?")
print("  • What to skip?")
print("  • How to format?")
print()
print("=" * 60)
print("\nYour agent is ready!")

## Session 1: Introduce Yourself

In [ ]:
# Customize this with YOUR preferences!

task = """
Create a morning brief system for me.

Repos I follow:
- [ADD YOUR REPOS HERE]
- Example: vLLM, PyTorch, Mini-Agent, etc.

Topics I care about:
- [ADD YOUR INTERESTS]
- Example: performance, new features, API changes, etc.

What to skip:
- [ADD FILTERS]
- Example: documentation, minor fixes, etc.

Generate the first brief and save to 'my_brief_v1.md'
"""

my_agent.add_user_message(task)
result = await my_agent.run()

## Check What Your Agent Remembered

In [ ]:
if my_memory_file.exists():
    print("What Your Agent Remembered:")
    print("=" * 60)
    my_memory = json.loads(my_memory_file.read_text())
    for i, note in enumerate(my_memory, 1):
        print(f"\n{i}. [{note.get('category', 'uncategorized')}]")
        print(f"   {note['content']}")
    print("\n" + "=" * 60)
else:
    print("No memory created yet")

## Session 2: Give Feedback (New Agent!)

In [ ]:
# Brand new agent - simulates new day
my_agent_day2 = Agent(
    llm_client=llm_client,
    system_prompt=system_prompt,
    tools=my_tools,
    max_steps=50,
    workspace_dir=str(workspace)
)

print("New agent created (simulating new day)")
print()

# Customize your feedback!
task2 = """
Generate my morning brief.

Additional feedback:
- [ADD YOUR IMPROVEMENTS]
- Example: add commit hashes, show authors, prioritize breaking changes, etc.

Save to 'my_brief_v2.md'
"""

my_agent_day2.add_user_message(task2)
result2 = await my_agent_day2.run()

## Session 3: Keep Learning!

In [ ]:
my_agent_day3 = Agent(
    llm_client=llm_client,
    system_prompt=system_prompt,
    tools=my_tools,
    max_steps=50,
    workspace_dir=str(workspace)
)

task3 = """
Generate my morning brief.

More improvements:
- [ADD MORE FEEDBACK]

Save to 'my_brief_v3.md'
"""

my_agent_day3.add_user_message(task3)
result3 = await my_agent_day3.run()

## Review Your Agent's Learning

In [ ]:
print("Your Agent's Complete Knowledge:")
print("=" * 60)

if my_memory_file.exists():
    my_memory = json.loads(my_memory_file.read_text())
    
    print(f"\nTotal notes: {len(my_memory)}")
    categories = set(note.get('category', 'uncategorized') for note in my_memory)
    print(f"Categories: {len(categories)}")
    
    for i, note in enumerate(my_memory, 1):
        print(f"\n{i}. [{note.get('category', 'uncategorized')}]")
        print(f"   {note['content']}")
    
    print("\n" + "=" * 60)
    print("\nYour agent has built a complete knowledge base!")
    print("Each session added more knowledge.")
else:
    print("No memory found")